# Building AI Application with Gemini 2.0

https://www.kdnuggets.com/building-ai-application-gemini-2

### Setting up

In [1]:
import os
from llama_index.llms.gemini import Gemini
from configparser import ConfigParser

c = ConfigParser()
c.read("C:\workspace\APIKEY_personal.ini")


os.environ["GOOGLE_API_KEY"] = c['KEY']["GEMINI"]

In [2]:


GoogleAPIKey = c['KEY']["GEMINI"]

llm = Gemini(
    model="models/gemini-2.0-flash-exp",
    api_key=GoogleAPIKey,
)


In [3]:
# test
response = llm.complete("Write a poem in the style of Rumi.")
print(response)

The dust of longing, a swirling dervish,
Kisses the hem of the Beloved's garment.
Forget the map, the compass, the purpose,
Lose yourself in the scent of jasmine, fervent.

The mind, a restless monkey, chattering,
Tries to define the ocean with a teacup.
Silence it, friend, let the heart be battering
Against the cage of reason, waking up.

For in the breaking, the true self blossoms,
A lotus rising from the muddy floor.
No need for prayers, for hymns, for solemn
Rituals. Just open the heart's door.

Let the Beloved rush in, a torrent of light,
Dissolving the ego, a fragile, thin ice.
Drown in the sweetness, the endless, pure might,
And find yourself, shattered, and twice as nice.

Don't seek the answer, the question is enough.
Embrace the mystery, the swirling, the unknown.
Let love be the wind, both gentle and rough,
And carry you home, where you've always been shown.



Load the embedding model


In [4]:
from llama_index.embeddings.gemini import GeminiEmbedding

embed_model = GeminiEmbedding(model_name='models/text-embedding-004')

Loading the documentation

In [5]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader('../data/raw/song_lyrics')
doc_txt = documents.load_data()

Building the Q&A Application

In [6]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 800
Settings.chunk_overlap = 20

c:\Users\TristramArmour\anaconda3\envs\learning\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in HuggingFaceInferenceAPIEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [7]:
from llama_index.core import VectorStoreIndex
from IPython.display import Markdown, display

index = VectorStoreIndex.from_documents(doc_txt, service_context=Settings)
index.storage_context.persist('./VectorStore')

In [9]:
query_engine = index.as_query_engine()
response = query_engine.query("Which verse do you think is the most thought-provoking by Rihanna? You have to choose one.")
display(Markdown(response.response))

I think the verse "You mistaken my love I brought for you for foundation, All that I wanted from you was to give me Something that I never had, Something that you've never seen, Something that you've never been!" is the most thought-provoking.


Building a RAG Chatbot with History

In [10]:
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.chat_engine import CondensePlusContextChatEngine

memory = ChatMemoryBuffer.from_defaults(token_limit=3900)

chat_engine = CondensePlusContextChatEngine.from_defaults(    
   index.as_retriever(),    
   memory=memory,    
   llm=llm,
)

response = chat_engine.chat(    
   "What do you think about Kanye West songs? "
)
display(Markdown(response.response))

Based on the documents, here's what I can tell you about Kanye West's songs:

*   **Variety of Themes:** His songs touch on various themes, from hustling and ambition ("Good Morning") to love and relationships, and even his own evolving persona.
*   **Introspection and Self-Awareness:** Some lyrics express a longing for an earlier version of himself ("I miss the old Kanye..."). This suggests a level of self-awareness and reflection in his music.
*   **Materialism and Success:** There are references to luxury brands like Louis Vuitton and celebrations with champagne, indicating themes of success and materialism.
*   **Controversy and Haters:** The lyrics acknowledge the presence of "haters" and address controversies, suggesting that Kanye is aware of his public image and the criticisms he faces.
*   **GOOD Music:** He frequently mentions "GOOD Music," highlighting his record label and the artists associated with it.
*   **Explicit Content:** Some lyrics contain explicit language and references to sexual themes.
*   **Musical Style:** There is a reference to "chop up the beats Kanye" which may refer to his sampling style.


In [11]:
response = chat_engine.stream_chat(    
   "Use one of the songs to write a poem. ",
)

for chunk in response.chat_stream:
    print(chunk.delta or "", end="", flush=True)

Okay, I can do that. Based on the provided lyrics from Kanye West's songs, here's a poem inspired by the themes and lines within:

**Oklahoma Dream**

Fifteen seconds fading, more to say,
They want me quiet, push my dreams away.
Advance in hand, a different scene,
Oklahoma calling, live at my aunt's, serene.

Romance the thought, leave it all behind,
Step from the light, the relentless grind.
Before the models, the bending low,
Before the Benz, the Rover's glow.

One beat for Hova, a simple plea,
Escape this sofa, finally be free.
The Chi's summer whispers, a choice to make,
Sell dreams or labor, for goodness sake.

Taco Bell churros, a fleeting score,
Europe's Euros, what are we living for?
They say you know it when it's gone, it's true,
But I got it now, what about you?

A store for MCs, inspiration's gleam,
No false promises, just a waking dream.
Flip like Anakin, a tragic fall,
Sell the mannequin, surrender all.

New bitch, Aniston, a fleeting phase,
Handle the panic, in a hazy daz